<a href="https://colab.research.google.com/github/syedmahmoodiagents/genai_classes/blob/main/Supervisor_Agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain-huggingface --q
!pip install langchain-community --q
!pip install wikipedia --q
!pip install langgraph --q
!pip install langgraph-supervisor --q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
  Preparing metadata (setup.py) ... done


In [2]:
import os

In [3]:
os.environ['HF_TOKEN'] = "hf_zIvXpRihcLDYQxQSByHOzDVeLf"

In [4]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

In [5]:
llm = ChatHuggingFace(llm = HuggingFaceEndpoint(repo_id="openai/gpt-oss-20b"))

In [6]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [8]:
wiki = WikipediaQueryRun(api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=480))

In [10]:
# wiki.invoke({"query": "Quantum computer"})

In [11]:
from langchain.agents import create_agent

In [18]:
agent_wiki = create_agent(
    model = llm,
    tools = [wiki],
    name = "search_agent",
    system_prompt="this tool is exclusively for searching a topic from internet"
)

In [12]:
from langchain.tools import tool

In [14]:
@tool
def add(a, b):
    """ this is for addition purpose only """
    return a + b

@tool
def multiply(a, b):
    """ this is for multiplication purpose only """
    return a * b

In [15]:
# add(10, 20)
# add.invoke(10,20) # once you have @tool installed

In [19]:
math_agent = create_agent(
    model=llm,
    tools=[add, multiply],
    name="algo_agent",
    system_prompt="You are a math agent. Always use one tool at a time"
)

In [20]:
last_agent = create_agent(
    model = llm,
    tools = [],
    name = "final_agent",
    system_prompt="if you feel that no other agent is helpful, then you can take help of this"
)

In [21]:
from langgraph_supervisor import create_supervisor

In [24]:
work_flow = create_supervisor(
    model = llm,
    agents = [agent_wiki, math_agent, last_agent],
    prompt=(
        "You are a team supervisor managing a research expert and a math expert"
        "For research events, use search_agent"
        "For math agent, use algo_agent"
        "if not sure with answers, use final_agent"
    )
)

In [27]:
super_agent = work_flow.compile()

In [28]:
result = super_agent.invoke({
    "messages": [{"role": "user", "content": "what is the population of France and then multiply by two?"}]
})

In [29]:
result

{'messages': [HumanMessage(content='what is the population of France and then multiply by two?', additional_kwargs={}, response_metadata={}, id='709f4f0e-1a2c-4e76-b6a7-3a2487ebc3aa'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"query":"population of France 2023","source":"news","topn":5}', 'name': 'transfer_to_search_agent', 'description': None}, 'id': 'fc_478ba6da-ddbb-4733-aeaa-e87976e49099', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 216, 'total_tokens': 292}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_228717f27c', 'finish_reason': 'tool_calls', 'logprobs': None}, name='supervisor', id='lc_run--01a00e4f-066b-7e51-9b2c-c64c859e7383-0', tool_calls=[{'name': 'transfer_to_search_agent', 'args': {'query': 'population of France 2023', 'source': 'news', 'topn': 5}, 'id': 'fc_478ba6da-ddbb-4733-aeaa-e87976e49099', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metada